In [22]:
import pandas as pd
import gpxpy
import os

## Configuration
Remplacez par le chemin vers les fichiers à traiter

In [ ]:
data_path = './input/AEHO001-data.csv'
track_path = './input/Navionics_archive_export_23-09-25_au_26-09-25.gpx'
instrument_name = 'AE_HO_001' # Suivre le format du numéro de série (owner_type_numéro)
output_dir = "output"

## Étape 1 : Charger les données GPX

In [24]:
def load_gpx(file_path):
    with open(file_path, 'r') as gpx_file:
        gpx = gpxpy.parse(gpx_file)
    points = []
    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                points.append({
                    'datetime': point.time,
                    'lat': point.latitude,
                    'long': point.longitude
                })
    
    df_gpx = pd.DataFrame(points)
    df_gpx['datetime'] = pd.to_datetime(df_gpx['datetime']).dt.tz_convert('UTC')  # Rendre naive
    return df_gpx

df_gpx = load_gpx(track_path)
df_gpx = df_gpx.set_index('datetime')

# Déterminer la plage de temps du GPX
gpx_start_at = df_gpx.index.min()
gpx_end_at = df_gpx.index.max()

print(f"Plage de temps du GPX : de {gpx_start_at} à {gpx_end_at}")

Plage de temps du GPX : de 2025-09-23 07:43:37.009000+00:00 à 2025-09-26 14:33:10.622000+00:00


## Étape 2 : Charger les données CSV

In [28]:
df = pd.read_csv(data_path, low_memory=False, parse_dates=[1], date_format="%m/%d/%Y %H:%M:%S", delimiter=';')

# Renommer les colonnes
df = df.rename(columns={
    'Date et heure (CEST)': 'datetime',
    'Température (W-CTD-00 22357924), °C': 'sea_temp',
    'Pression absolue (W-CTD-00 22357924), kPa': 'air_pres',
    'Conductivité électrique (W-CTD-00 22357924), µS/cm': 'ec_abs',
    'Conductivité spécifique (W-CTD-00 22357924), µS/cm': 'ec25',
    'Salinité (W-CTD-00 22357924), PSU': 'sea_sal',
    'Matières solides dissoutes totales (W-CTD-00 22357924), mg/L': 'total_dissolved_solids',
})

# Supprimer les colonnes inutiles
df = df.drop(
    columns=['#', 'Hôte connecté', 'Fin du fichier', 'Démarré', 'Arrêté', 'Bouton Haut', 'Bouton Bas', 'Détection d’eau'],
    errors='ignore'
)

print(df.head())

# Convertir le fuseau horaire de CEST en UTC
df['datetime'] = df['datetime'].dt.tz_localize('Europe/Paris')
df['datetime'] = df['datetime'].dt.tz_convert('UTC')
df = df.set_index('datetime')

# Convertir la pression absolue de kPa en hPa
df['air_pres'] = df['air_pres'] * 10
df['air_pres'] = df['air_pres'].round(2)

             datetime  sea_temp  air_pres   ec_abs     ec25  sea_sal  \
0 2025-07-21 18:44:42     11.32   102.380  35819.3  50262.0    31.53   
1 2025-07-21 18:45:42     11.29   102.379  35788.2  50254.4    31.52   
2 2025-07-21 18:46:42     11.27   102.338  35683.2  50138.5    31.44   
3 2025-07-21 18:47:42     11.25   102.296  35686.9  50179.7    31.46   
4 2025-07-21 18:48:42     11.22   102.336  35638.0  50147.0    31.43   

   total_dissolved_solids  
0                23282.52  
1                23262.33  
2                23194.10  
3                23196.49  
4                23164.71  


In [29]:
# df = df[(df.name >= min_time_gpx) & (df.name <= max_time_gpx)]
print("Plage de dates CSV :", df.index.min(), "à", df.index.max())
print("Plage de dates CSV :", df.index.min(), "à", df.index.max())

Plage de dates CSV : 2025-07-21 16:44:42+00:00 à 2025-09-26 15:37:28+00:00
Plage de dates CSV : 2025-07-21 16:44:42+00:00 à 2025-09-26 15:37:28+00:00


## Étape 3 : Interpolation de la position des mesures à partir des traces GPS  

In [30]:
def interpolate_position(row, df_gpx):
    t = row.name
    # Trouver les deux points GPX encadrant t
    idx = df_gpx.index.searchsorted(t, side='right') - 1
    if idx < 0 or idx >= len(df_gpx) - 1:
        return None, None  # Hors des limites

    point_A = df_gpx.iloc[idx]
    point_B = df_gpx.iloc[idx + 1]

    tA, tB = point_A.name, point_B.name
    latA, longA = point_A['lat'], point_A['long']
    latB, longB = point_B['lat'], point_B['long']

    ratio = (t - tA).total_seconds() / (tB - tA).total_seconds()
    lat = latA + (latB - latA) * ratio
    long = longA + (longB - longA) * ratio

    return lat, long

In [31]:
df[['lat', 'long']] = df.apply(lambda row: interpolate_position(row, df_gpx), axis=1, result_type='expand')
df = df.dropna(subset=['lat', 'long']) # Nettoyer les valeurs manquantes

## Étape 4 : Exporter les données

In [32]:
os.makedirs(output_dir, exist_ok=True)

start_at = df.index.min().strftime('%Y-%m-%d_%H:%M:%S')
output_csv_final = os.path.join(output_dir, f"{instrument_name}_{start_at}_with_positions.csv")
df.to_csv(output_csv_final, index=False)